In [ ]:
import osmnx as ox
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import esda
import spreg
from libpysal import weights
from splot.esda import lisa_cluster
import statsmodels.api as sm
import matplotlib as mpl
import json
import seaborn as sns
from gwlearn.linear_model import GWLinearRegression
from sklearn import metrics
from mgwr.gwr import GWR

In [ ]:
with open("../scripts/08_plot_delays_politics.json", "r") as f:
    plot_params = json.load(f)
for key in plot_params["rcparams"]:
    mpl.rcParams[key] = plot_params["rcparams"][key]

In [ ]:
G = ox.load_graphml(
    "../data/processed/paris_simplified_results/paris_cleaned_multigraph.graphml"
)
gdf_edges = ox.graph_to_gdfs(G, edges=True, nodes=False)
gdf_edges = gdf_edges[["geometry", "length", "built"]]

In [ ]:
gdf_iris = gpd.read_file(
    "../data/processed/paris_official_data/paris_dem_iris_2021_condensed_enriched.gpkg"
)

In [ ]:
gdf_iris = gdf_iris.to_crs(gdf_edges.crs)

In [ ]:
BUSINESSES_TO_COUNT = [
    "Alimentaire",
    "Grand magasin",
    "Hôtel",
    "Non Alimentaire",
    "Restauration",
    "Service commercial",
]

In [ ]:
gdf_iris.drop(
    ["CODE_IRIS", "geometry", "population", "active_population", "poverty_rate"]
    + BUSINESSES_TO_COUNT
    + [bus + "_density" for bus in BUSINESSES_TO_COUNT],
    axis=1,
).corr()

In [ ]:
gdf_vote_sta = gpd.read_file(
    "../data/processed/paris_official_data/paris_vote_list_2020.gpkg"
)
gdf_vote_sta = gdf_vote_sta.to_crs(gdf_edges.crs)

In [ ]:
w = weights.Queen.from_dataframe(gdf_vote_sta, use_index=False)
w.transform = "R"
moran = esda.moran.Moran_Local(gdf_vote_sta["LUG_share"], w)
fig = lisa_cluster(moran, gdf_vote_sta)
plt.show()

In [ ]:
share = {}
total = gdf_vote_sta["NB_EXPRIM"].sum()
for label in gdf_vote_sta.columns[:9]:
    share[label] = round(100 * (gdf_vote_sta[label].sum() / total), 1)
share

In [ ]:
gdf_vote_sta_2026 = gpd.read_file(
    "../data/processed/paris_simplified_results/paris_vote_sta_2026_bikenet.gpkg"
)

In [ ]:
share = {}
total = gdf_vote_sta_2026["Votants"].sum()
for label in gdf_vote_sta_2026.columns[8:-12]:
    share[label] = round(100 * (gdf_vote_sta_2026[label].sum() / total), 1)
share

In [ ]:
gdf_vote_arr = gpd.read_file(
    "../data/processed/paris_simplified_results/paris_vote_arr_2020_bikenet.gpkg"
)

In [ ]:
gdf_vote_arr

In [ ]:
rm = gdf_vote_arr["LUD_share"].mean()
lm = gdf_vote_arr["LUG_share"].mean()
vote_avg_ratio = rm - lm
gdf_vote_arr["ratio_lr"] = (
    (gdf_vote_arr["LUD"] - gdf_vote_arr["LUG"]) / gdf_vote_arr["NB_EXPRIM"]
) - vote_avg_ratio

In [ ]:
vote_avg_ratio

# Arrondissements results

In [ ]:
gdf_vote_arr.plot(column="ratio_lr", legend=True, cmap="bwr_r", vmin=-0.9, vmax=0.9)

In [ ]:
bin_edges = np.linspace(
    gdf_vote_arr["ratio_lr"].min(), gdf_vote_arr["ratio_lr"].max(), num=6
)

In [ ]:
def put_in_bin(x, bins):
    for i in range(len(bins) - 1):
        if x >= bins[i] and x < bins[i + 1]:
            return round((bins[i] + bins[i + 1]) / 2, 2)

In [ ]:
bin_edges = np.linspace(
    min(gdf_vote_arr[["LUG_share", "LUD_share"]].min()),
    max(gdf_vote_arr[["LUG_share", "LUD_share"]].max()),
    num=5,
)
gdf_vote_arr["binned_LUG_share"] = gdf_vote_arr["LUG_share"].apply(
    lambda x: put_in_bin(x, bin_edges)
)
gdf_vote_arr["binned_LUD_share"] = gdf_vote_arr["LUD_share"].apply(
    lambda x: put_in_bin(x, bin_edges)
)
fig, ax = plt.subplots()
sns.swarmplot(
    gdf_vote_arr,
    x="binned_LUG_share",
    y="length_accomplished_share",
    color="red",
    ax=ax,
)
sns.swarmplot(
    gdf_vote_arr,
    x="binned_LUD_share",
    y="length_accomplished_share",
    color="blue",
    ax=ax,
)
ax.set_xlabel("Share vote for the party")

In [ ]:
bin_edges = np.linspace(
    gdf_vote_arr["ratio_lr"].min(), gdf_vote_arr["ratio_lr"].max(), num=6
)
gdf_vote_arr["binned_ratio"] = gdf_vote_arr["ratio_lr"].apply(
    lambda x: put_in_bin(x, bin_edges)
)
sns.swarmplot(gdf_vote_arr, x="binned_ratio", y="length_accomplished_share")

In [ ]:
sns.boxplot(gdf_vote_arr, x="binned_ratio", y="length_accomplished_share")

In [ ]:
sns.violinplot(gdf_vote_arr, x="binned_ratio", y="length_accomplished_share", cut=0)

In [ ]:
gdf_iris

In [ ]:
model = sm.OLS(
    gdf_vote_arr["length_accomplished_share"],
    sm.add_constant(gdf_vote_arr["ratio_LR"].values),
).fit()
model.summary()

In [ ]:
model = sm.OLS(
    gdf_vote_arr["length_accomplished_share"],
    sm.add_constant(gdf_vote_arr["ratio_LR"].values),
).fit(cov_type="HC1")
model.summary()

In [ ]:
model = sm.OLS(
    gdf_vote_arr["length_accomplished_share"],
    sm.add_constant(gdf_vote_arr["ratio_LR"].values),
).fit(cov_type="HC2")
model.summary()

In [ ]:
model = sm.OLS(
    gdf_vote_arr["length_accomplished_share"],
    sm.add_constant(gdf_vote_arr["ratio_LR"].values),
).fit(cov_type="HC3")
model.summary()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(gdf_vote_arr["ratio_LR"], model.u, color="black")
ax.plot([-1, 1], [0, 0], color="gray", linestyle="dashed")
ax.set_xlim([-0.5, 0.9])

In [ ]:
gdf_vote_arr["residuals"] = model.u
gdf_vote_arr.plot(column="residuals")

In [ ]:
model = spreg.GM_Lag(
    gdf_vote_arr["length_accomplished_share"],
    gdf_vote_arr["ratio_LR"],
    w=weights.Queen.from_dataframe(gdf_vote_arr),
)
print(model.summary)

In [ ]:
gdf_vote_sta = gpd.read_file(
    "../data/processed/paris_simplified_results/paris_vote_sta_2020_bikenet.gpkg"
)

In [ ]:
BINS_ARR = 4
bin_edges = np.linspace(
    min(gdf_vote_sta[["LUG_share", "LUD_share"]].min()),
    max(gdf_vote_sta[["LUG_share", "LUD_share"]].max()),
    num=BINS_ARR + 1,
)
mean_acc = gdf_vote_sta["length_accomplished_share_smoothed"].mean()

In [ ]:
SEED = 7
rng = np.random.default_rng(seed=SEED)
SHUFFLE_NUMBER = 100
res_arr_rw = []
res_arr_lw = []
for i in range(SHUFFLE_NUMBER):
    gdf_shuffled = gdf_vote_sta.copy()
    gdf_shuffled["length_accomplished_share_smoothed"] = rng.permutation(
        gdf_shuffled["length_accomplished_share_smoothed"].values
    )
    res_arr_rw.append(
        [
            gdf_shuffled[
                (gdf_shuffled["LUD_share"] >= bin_edges[i])
                & (gdf_shuffled["LUD_share"] < bin_edges[i + 1])
            ]["length_accomplished_share_smoothed"].mean()
            for i in range(len(bin_edges) - 1)
        ]
    )
    res_arr_lw.append(
        [
            gdf_shuffled[
                (gdf_shuffled["LUG_share"] >= bin_edges[i])
                & (gdf_shuffled["LUG_share"] < bin_edges[i + 1])
            ]["length_accomplished_share_smoothed"].mean()
            for i in range(len(bin_edges) - 1)
        ]
    )

In [ ]:
fig, ax = plt.subplots(figsize=plot_params["figsize"])
for ids in range(len(plot_params["party"])):
    res = [
        gdf_vote_sta[
            (gdf_vote_sta[plot_params["party"][ids]] >= bin_edges[i])
            & (gdf_vote_sta[plot_params["party"][ids]] < bin_edges[i + 1])
        ]["length_accomplished_share_smoothed"].mean()
        for i in range(len(bin_edges) - 1)
    ]
    err = [
        gdf_vote_sta[
            (gdf_vote_sta[plot_params["party"][ids]] >= bin_edges[i])
            & (gdf_vote_sta[plot_params["party"][ids]] < bin_edges[i + 1])
        ]["length_accomplished_share_smoothed"].std()
        for i in range(len(bin_edges) - 1)
    ]
    ax.errorbar(
        [(bin_edges[i] + bin_edges[i + 1]) / 2 for i in range(len(bin_edges) - 1)],
        res,
        yerr=err,
        **{
            key: val[ids]
            for key, val in plot_params.items()
            if key not in ["figsize", "rcparams", "party", "cmap", "scheme"]
        },
    )
# ax.errorbar(
#     [
#         (bin_edges[i] + bin_edges[i + 1]) / 2
#         for i in range(len(bin_edges) - 1)
#     ],
#     np.mean(res_arr_rw, axis=0),
#     yerr=[np.percentile(res_arr_rw, q=2.5, axis=0), np.percentile(res_arr_rw, q=97.5, axis=0)],
#     color="lightblue",
#     label="reshuffled_LUD"
# )
# ax.errorbar(
#     [
#         (bin_edges[i] + bin_edges[i + 1]) / 2
#         for i in range(len(bin_edges) - 1)
#     ],
#     np.mean(res_arr_lw, axis=0),
#     yerr=[np.percentile(res_arr_lw, q=2.5, axis=0), np.percentile(res_arr_lw, q=97.5, axis=0)],
#     color="firebrick",
#     label="reshuffled_LUG"
# )
yy = [mean_acc, mean_acc]
ax.plot(
    [bin_edges[0], bin_edges[-1]],
    yy,
    linestyle="dashed",
    color="#E1E1E1",
    zorder=0,
    label="Average accomplishment",
)
# TODO add legend modification
ax.legend()
ax.set_xlabel("Share of votes for the party")
ylabel = "Share of bicycle lanes accomplished"
ax.set_ylabel(ylabel)
ax.set_xlim([0.05, 0.75])
ylim = [-0.3, 0.3]
ylim += mean_acc
ax.set_ylim(ylim)

In [ ]:
gdf_vote_arr

In [ ]:
X = gdf_vote_arr[["ratio_lr"]]
Y = gdf_vote_arr["length_accomplished_share"]
geometry = gdf_vote_arr.geometry.representative_point()
adaptive = GWLinearRegression(bandwidth=20, fixed=False)
adaptive.fit(
    X,
    Y,
    geometry=geometry,
)

In [ ]:
metrics.r2_score(gdf_vote_arr["length_accomplished_share"], adaptive.pred_)

In [ ]:
gdf_vote_arr.plot(adaptive.local_r2_, legend=True).set_axis_off()

In [ ]:
gdf_vote_arr.plot(adaptive.resid_, legend=True).set_axis_off()

In [ ]:
adaptive.local_coef_["ratio_lr"].mean()

In [ ]:
extremum = max(
    abs(adaptive.local_coef_["ratio_lr"].min()),
    abs(adaptive.local_coef_["ratio_lr"].max()),
)
gdf_vote_arr.plot(
    adaptive.local_coef_["ratio_lr"],
    legend=True,
    cmap="bwr",
    vmin=-extremum,
    vmax=extremum,
).set_axis_off()

In [ ]:
fig, ax = plt.subplots()
gdf_vote_arr.plot(ax=ax)
gdf_vote_arr.geometry.representative_point().plot(ax=ax, color="black")

In [ ]:
mg = GWR(
    geometry.get_coordinates().values,
    Y.values.reshape(-1, 1),
    X.values,
    bw=20,
    fixed=False,
    kernel="gaussian",
).fit()

print(f"mgwr R²: {metrics.r2_score(Y, mg.predy):.4f}  AICc: {mg.aicc:.1f}")

# Voting station results

In [ ]:
BINS_ARR = 4
bin_edges = np.linspace(
    min(gdf_vote_sta[["LUG_share", "LUD_share"]].min()),
    max(gdf_vote_sta[["LUG_share", "LUD_share"]].max()),
    num=BINS_ARR + 1,
)

In [ ]:
rm = gdf_vote_sta["LUD_share"].mean()
lm = gdf_vote_sta["LUG_share"].mean()
vote_avg_ratio = rm - lm
gdf_vote_sta["ratio_lr"] = (
    (gdf_vote_sta["LUD"] - gdf_vote_sta["LUG"]) / gdf_vote_sta["NB_EXPRIM"]
) - vote_avg_ratio

In [ ]:
fig, ax = plt.subplots(figsize=plot_params["figsize"])
gdf_vote_sta.plot(column="ratio_lr", legend=True, cmap="bwr_r", vmin=-1, vmax=1, ax=ax)
ax.axis("off")

In [ ]:
fig, ax = plt.subplots()
mean_acc = gdf_vote_sta["length_accomplished_share_smoothed"].mean()
bin_edges = np.linspace(
    gdf_vote_sta["ratio_lr"].min(), gdf_vote_sta["ratio_lr"].max(), num=6
)
res = [
    gdf_vote_sta[
        (gdf_vote_sta["ratio_lr"] >= bin_edges[i])
        & (gdf_vote_sta["ratio_lr"] < bin_edges[i + 1])
    ]["length_accomplished_share_smoothed"].mean()
    for i in range(len(bin_edges) - 1)
]
err = [
    gdf_vote_sta[
        (gdf_vote_sta["ratio_lr"] >= bin_edges[i])
        & (gdf_vote_sta["ratio_lr"] < bin_edges[i + 1])
    ]["length_accomplished_share_smoothed"].sem()
    for i in range(len(bin_edges) - 1)
]
ax.errorbar(
    [(bin_edges[i] + bin_edges[i + 1]) / 2 for i in range(len(bin_edges) - 1)],
    res,
    yerr=err,
    marker="o",
    color="black",
)
ax.plot(
    [-0.9, 0.9],
    [mean_acc, mean_acc],
    linestyle="dashed",
    color="#E1E1E1",
    zorder=0,
)
ax.set_ylabel("Share accomplished")
ax.set_xlabel("Left-wing / Right-wing")
ax.set_xlim([-0.9, 0.9])

In [ ]:
model = sm.OLS(
    gdf_vote_sta["length_accomplished_share_smoothed"],
    sm.add_constant(gdf_vote_sta["ratio_lr"].values),
).fit()
model.summary()

In [ ]:
fig, ax = plt.subplots(figsize=plot_params["figsize"])
ax.scatter(
    gdf_vote_sta["ratio_lr"],
    gdf_vote_sta["length_accomplished_share_smoothed"],
    color="black",
    s=10,
)
xx = np.linspace(
    gdf_vote_sta["ratio_lr"].min(), gdf_vote_sta["ratio_lr"].max(), num=100
)
ax.plot(xx, model.params["x1"] * xx + model.params["const"], color="red")
ax.set_ylabel("Share accomplished")
ax.set_xlabel("Left-wing / Right-wing")

In [ ]:
bin_edges = np.linspace(
    gdf_vote_sta["ratio_lr"].min(), gdf_vote_sta["ratio_lr"].max(), num=6
)
gdf_vote_sta["binned_ratio"] = gdf_vote_sta["ratio_lr"].apply(
    lambda x: put_in_bin(x, bin_edges)
)
sns.swarmplot(
    gdf_vote_sta, x="binned_ratio", y="length_accomplished_share_smoothed", size=2
)

In [ ]:
sns.violinplot(
    gdf_vote_sta, x="binned_ratio", y="length_accomplished_share_smoothed", cut=0
)

In [ ]:
sns.boxplot(gdf_vote_sta, x="binned_ratio", y="length_accomplished_share_smoothed")

# Change vote

In [ ]:
gdf_vote_arr_2026 = gpd.read_file(
    "../data/processed/paris_simplified_results/paris_vote_arr_2026_bikenet.gpkg"
)
gdf_comparison_2020 = gdf_vote_arr.copy()
gdf_comparison_2020["LW_share"] = (
    gdf_comparison_2020["LUG_share"]
    + gdf_comparison_2020["LFI_share"]
    + gdf_comparison_2020["LDIV_share"]
    + gdf_comparison_2020["LVEC_share"]
)
gdf_comparison_2020["RW_share"] = (
    gdf_comparison_2020["LUD_share"]
    + gdf_comparison_2020["LRN_share"]
    + gdf_comparison_2020["LDVD_share"]
    + gdf_comparison_2020["LUC_share"]
    + gdf_comparison_2020["LDVC_share"]
)
gdf_comparison_2020 = gdf_comparison_2020[
    ["LW_share", "RW_share", "length_accomplished_share", "geometry", "NUM_ARROND"]
]
gdf_comparison_2020["LR_ratio"] = (
    gdf_comparison_2020["RW_share"] - gdf_comparison_2020["LW_share"]
)
gdf_comparison_2020["LR_ratio_norm"] = gdf_comparison_2020["LR_ratio"] - (
    gdf_comparison_2020["RW_share"].mean() - gdf_comparison_2020["LW_share"].mean()
)
gdf_comparison_2020["index"] = gdf_comparison_2020.index
gdf_comparison_2026 = gdf_vote_arr_2026.copy()
gdf_comparison_2026["LW_share"] = (
    gdf_comparison_2026["LUG_share"]
    + gdf_comparison_2026["LFI_share"]
    + gdf_comparison_2026["LDVG_share"]
    + gdf_comparison_2026["LEXG_share"]
)
gdf_comparison_2026["RW_share"] = (
    gdf_comparison_2026["LUD_share"]
    + gdf_comparison_2026["LEXD_share"]
    + gdf_comparison_2026["LRN_share"]
    + gdf_comparison_2026["LUXD_share"]
    + gdf_comparison_2026["LDVD_share"]
    + gdf_comparison_2026["LDVC_share"]
    + gdf_comparison_2026["LUC_share"]
)
gdf_comparison_2026 = gdf_comparison_2026[
    ["LW_share", "RW_share", "length_accomplished_share", "geometry", "arrondissement"]
]
gdf_comparison_2026["LR_ratio"] = (
    gdf_comparison_2026["RW_share"] - gdf_comparison_2026["LW_share"]
)
gdf_comparison_2026["LR_ratio_norm"] = gdf_comparison_2026["LR_ratio"] - (
    gdf_comparison_2026["RW_share"].mean() - gdf_comparison_2026["LW_share"].mean()
)
gdf_comparison_2026["index"] = gdf_comparison_2026.index
gdf_comparison_2026["arrondissement"] = gdf_comparison_2026["arrondissement"].map(int)

In [ ]:
gdf_comparison = gdf_comparison_2020.merge(
    gdf_comparison_2026,
    left_on="NUM_ARROND",
    right_on="arrondissement",
    suffixes=["_2020", "_2026"],
)
gdf_comparison["geometry"] = gdf_comparison["geometry_2020"]
gdf_comparison["change_vote"] = (
    gdf_comparison["LR_ratio_2026"] - gdf_comparison["LR_ratio_2020"]
)
gdf_comparison["change_vote_norm"] = (
    gdf_comparison["LR_ratio_norm_2026"] - gdf_comparison["LR_ratio_norm_2020"]
)

In [ ]:
gdf_comparison[["change_vote", "length_accomplished_share_2020"]].corr()

In [ ]:
fig, ax = plt.subplots()
gdf_comparison.plot(
    ax=ax, column="change_vote_norm", cmap="bwr_r", legend=True, vmin=-0.5, vmax=0.5
)
ax.axis("off")

In [ ]:
model = sm.OLS(
    gdf_comparison["length_accomplished_share_2020"],
    sm.add_constant(gdf_comparison["change_vote"]),
    hasconst=True,
).fit()
model.summary()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    gdf_comparison["change_vote"],
    gdf_comparison["length_accomplished_share_2020"],
    color="black",
    s=10,
)
xx = np.linspace(
    gdf_comparison["change_vote"].min(), gdf_comparison["change_vote"].max(), num=100
)
ax.plot(xx, xx * model.params["change_vote"] + model.params["const"], color="red")

In [ ]:
gdf_comparison_2020 = gdf_vote_sta.copy()
gdf_comparison_2020["LW_share"] = (
    gdf_comparison_2020["LUG_share"]
    + gdf_comparison_2020["LFI_share"]
    + gdf_comparison_2020["LDIV_share"]
    + gdf_comparison_2020["LVEC_share"]
)
gdf_comparison_2020["RW_share"] = (
    gdf_comparison_2020["LUD_share"]
    + gdf_comparison_2020["LRN_share"]
    + gdf_comparison_2020["LDVD_share"]
    + gdf_comparison_2020["LUC_share"]
    + gdf_comparison_2020["LDVC_share"]
)
gdf_comparison_2020 = gdf_comparison_2020[
    ["LW_share", "RW_share", "length_accomplished_share_smoothed", "geometry"]
]
gdf_comparison_2020["LR_ratio"] = (
    gdf_comparison_2020["RW_share"] - gdf_comparison_2020["LW_share"]
)
gdf_comparison_2020["LR_ratio_norm"] = gdf_comparison_2020["LR_ratio"] - (
    gdf_comparison_2020["RW_share"].mean() - gdf_comparison_2020["LW_share"].mean()
)
gdf_comparison_2020["index"] = gdf_comparison_2020.index

In [ ]:
gdf_comparison_2026 = gdf_vote_sta_2026.copy()
gdf_comparison_2026["LW_share"] = (
    gdf_comparison_2026["LUG_share"]
    + gdf_comparison_2026["LFI_share"]
    + gdf_comparison_2026["LDVG_share"]
    + gdf_comparison_2026["LEXG_share"]
)
gdf_comparison_2026["RW_share"] = (
    gdf_comparison_2026["LUD_share"]
    + gdf_comparison_2026["LEXD_share"]
    + gdf_comparison_2026["LRN_share"]
    + gdf_comparison_2026["LUXD_share"]
    + gdf_comparison_2026["LDVD_share"]
    + gdf_comparison_2026["LDVC_share"]
    + gdf_comparison_2026["LUC_share"]
)
gdf_comparison_2026 = gdf_comparison_2026[
    ["LW_share", "RW_share", "length_accomplished_share_smoothed", "geometry"]
]
gdf_comparison_2026["LR_ratio"] = (
    gdf_comparison_2026["RW_share"] - gdf_comparison_2026["LW_share"]
)
gdf_comparison_2026["LR_ratio_norm"] = gdf_comparison_2026["LR_ratio"] - (
    gdf_comparison_2026["RW_share"].mean() - gdf_comparison_2026["LW_share"].mean()
)
gdf_comparison_2026["index"] = gdf_comparison_2026.index

In [ ]:
gdf_comparison = gpd.overlay(
    gdf_comparison_2020, gdf_comparison_2026, how="intersection", keep_geom_type=False
)
gdf_comparison["area"] = gdf_comparison.geometry.area
gdf_comparison.sort_values(by="area", inplace=True)
gdf_comparison.drop_duplicates(subset="index_1", keep="last", inplace=True)

In [ ]:
gdf_comparison["change_vote"] = (
    gdf_comparison["LR_ratio_2"] - gdf_comparison["LR_ratio_1"]
)
gdf_comparison["change_vote_norm"] = (
    gdf_comparison["LR_ratio_norm_2"] - gdf_comparison["LR_ratio_norm_1"]
)

In [ ]:
gdf_comparison[
    ["change_vote", "change_vote_norm", "length_accomplished_share_smoothed_1"]
].corr()

In [ ]:
fig, ax = plt.subplots()
gdf_comparison.plot(
    ax=ax, column="change_vote_norm", cmap="bwr_r", legend=True, vmin=-0.5, vmax=0.5
)
ax.axis("off")

In [ ]:
model = sm.OLS(
    gdf_comparison["length_accomplished_share_smoothed_1"],
    sm.add_constant(gdf_comparison["change_vote"]),
    hasconst=True,
).fit()
model.summary()

In [ ]:
fig, ax = plt.subplots()
ax.scatter(
    gdf_comparison["change_vote"],
    gdf_comparison["length_accomplished_share_smoothed_1"],
    color="black",
    s=10,
)
xx = np.linspace(
    gdf_comparison["change_vote"].min(), gdf_comparison["change_vote"].max(), num=100
)
ax.plot(xx, xx * model.params["change_vote"] + model.params["const"], color="red")

In [ ]:
gdf_comparison["length_accomplished_binary"] = gdf_comparison[
    "length_accomplished_share_smoothed_1"
].apply(lambda x: 1 if x >= 0.5 else 0)

In [ ]:
gdf_comparison["More_LW"] = gdf_comparison["change_vote"].apply(
    lambda x: 1 if x < 0 else 0
)
gdf_comparison["More_RW"] = gdf_comparison["change_vote"].apply(
    lambda x: 1 if x > 0 else 0
)

In [ ]:
fig, ax = plt.subplots()
ax.bar(
    ["More_LW", "More_RW"],
    [
        gdf_comparison[gdf_comparison["More_LW"] == 1][
            "length_accomplished_share_smoothed_1"
        ].mean(),
        gdf_comparison[gdf_comparison["More_RW"] == 1][
            "length_accomplished_share_smoothed_1"
        ].mean(),
    ],
    color=["red", "blue"],
)
ax.set_ylabel("Length accomplished share")

In [ ]:
model = sm.Logit(
    gdf_comparison["More_LW"],
    gdf_comparison["length_accomplished_share_smoothed_1"],
).fit()
model.summary()